# 📦 Notebook 1 — Data Loading & Preparation

**Goal:** Load the raw petrol pump Excel data, understand its structure,
clean it, fix data types, and save a clean version for all future notebooks.

**Steps in this notebook:**
1. Load the Excel file
2. Inspect shape, columns, types
3. Check for missing values
4. Fix data types (dates, integers)
5. Validate business rules (stock logic)
6. Save clean CSV for downstream notebooks

In [1]:
# ── Install required libraries (run once) ──────────────────────
# !pip install pandas openpyxl matplotlib seaborn scikit-learn imbalanced-learn xgboost

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

Libraries loaded ✅


## Step 1 — Load the Excel file

In [3]:
# ── Load data ──────────────────────────────────────────────────
# Update path to your actual file location
FILE_PATH = '../data/petrol_pump_2011_2025.xlsx'

df = pd.read_excel(FILE_PATH)

print(f'Rows    : {df.shape[0]}')
print(f'Columns : {df.shape[1]}')
print(f'\nColumn names:')
for c in df.columns:
    print(f'  {c}')

Rows    : 5189
Columns : 14

Column names:
  Date
  Day
  Opening_Stock
  MS_Sold
  HSD1_Sold
  HSD2_Sold
  HSD3_Sold
  Total_Sold
  Closing_Stock
  Cash
  Online
  Card
  Dip
  Refill_Required


## Step 2 — Inspect structure

In [4]:
# First 5 rows
df.head()

,Date,Day,Opening_Stock,MS_Sold,HSD1_Sold,HSD2_Sold,HSD3_Sold,Total_Sold,Closing_Stock,Cash,Online,Card,Dip,Refill_Required
0,01-01-2011,Saturday,12000,711,1881,1702,1052,5346,6654,227287,158865,99086,55,No
1,02-01-2011,Sunday,6654,757,2174,1748,1155,5834,820,263097,203622,112244,6,Yes
2,03-01-2011,Monday,12000,455,1625,1302,693,4075,7925,170856,115783,73403,66,No
3,04-01-2011,Tuesday,7925,409,1430,1037,902,3778,4147,148728,130226,75541,34,No
4,05-01-2011,Wednesday,4147,476,1528,1251,680,3935,212,172731,136389,78490,1,Yes


In [5]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5189 entries, 0 to 5188
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Date             5189 non-null   object
 1   Day              5189 non-null   object
 2   Opening_Stock    5189 non-null   int64 
 3   MS_Sold          5189 non-null   int64 
 4   HSD1_Sold        5189 non-null   int64 
 5   HSD2_Sold        5189 non-null   int64 
 6   HSD3_Sold        5189 non-null   int64 
 7   Total_Sold       5189 non-null   int64 
 8   Closing_Stock    5189 non-null   int64 
 9   Cash             5189 non-null   int64 
 10  Online           5189 non-null   int64 
 11  Card             5189 non-null   int64 
 12  Dip              5189 non-null   int64 
 13  Refill_Required  5189 non-null   object
dtypes: int64(11), object(3)
memory usage: 567.7+ KB


In [6]:
# Basic statistics for all numeric columns
df.describe().round(2)

,Opening_Stock,MS_Sold,HSD1_Sold,HSD2_Sold,HSD3_Sold,Total_Sold,Closing_Stock,Cash,Online,Card,Dip
count,5189.00,5189.00,5189.00,5189.00,5189.00,5189.00,5189.00,5189.00,5189.00,5189.00,5189.00
mean,8320.72,625.44,1982.88,1565.88,1042.34,5216.55,3474.76,224358.29,164204.25,112006.60,28.82
std,3436.00,122.63,348.19,279.55,228.07,885.06,2951.56,39815.09,29736.76,22084.18,24.19
min,2000.00,319.00,1051.00,866.00,456.00,2887.00,0.00,121322.00,88914.00,55064.00,1.00
25%,5946.00,538.00,1729.00,1361.00,876.00,4577.00,246.00,195439.00,142705.00,96327.00,2.00
50%,7473.00,614.00,1957.00,1540.00,1023.00,5133.00,2833.00,221113.00,161637.00,109846.00,23.00
75%,12000.00,703.00,2209.00,1747.00,1189.00,5792.00,6489.00,250089.00,182919.00,125704.00,54.00
max,12000.00,1045.00,3071.00,2457.00,1958.00,7500.00,8873.00,354489.00,272001.00,191232.00,73.00


## Step 3 — Check for missing values

In [7]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

if df.isnull().sum().sum() == 0:
    print('✅ No missing values found!')
else:
    print('⚠️  Missing values found — filling with forward fill')
    df.fillna(method='ffill', inplace=True)

                 Missing Count  Missing %
Date                         0        0.0
Day                          0        0.0
Opening_Stock                0        0.0
MS_Sold                      0        0.0
HSD1_Sold                    0        0.0
HSD2_Sold                    0        0.0
HSD3_Sold                    0        0.0
Total_Sold                   0        0.0
Closing_Stock                0        0.0
Cash                         0        0.0
Online                       0        0.0
Card                         0        0.0
Dip                          0        0.0
Refill_Required              0        0.0

Total missing values: 0
✅ No missing values found!


## Step 4 — Fix data types

In [8]:
# Convert Date column to proper datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Extract useful date features
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['DayOfWeek']  = df['Date'].dt.dayofweek          # 0=Monday, 6=Sunday
df['DayOfYear']  = df['Date'].dt.dayofyear
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['Quarter']    = df['Date'].dt.quarter
df['Is_Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Encode Day name as number (already have DayOfWeek but keeping Day as string too)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['Day_Num'] = df['Day'].map({d: i for i, d in enumerate(day_order)})

# Encode target: Refill_Required → 0/1
df['Target'] = (df['Refill_Required'] == 'Yes').astype(int)

print('Date features added:')
print(df[['Date','Year','Month','DayOfWeek','Is_Weekend','Quarter','Target']].head(10))

Date features added:
        Date  Year  Month  DayOfWeek  Is_Weekend  Quarter  Target
0 2011-01-01  2011      1          5           1        1       0
1 2011-01-02  2011      1          6           1        1       1
2 2011-01-03  2011      1          0           0        1       0
3 2011-01-04  2011      1          1           0        1       0
4 2011-01-05  2011      1          2           0        1       1
5 2011-01-06  2011      1          3           0        1       0
6 2011-01-07  2011      1          4           0        1       0
7 2011-01-08  2011      1          5           1        1       1
8 2011-01-09  2011      1          6           1        1       0
9 2011-01-10  2011      1          0           0        1       1


## Step 5 — Validate business rules

In [10]:
print('=== Business Rule Validation ===')

# Rule 1: Total_Sold = MS + HSD1 + HSD2 + HSD3
df['_calc_total'] = df['MS_Sold'] + df['HSD1_Sold'] + df['HSD2_Sold'] + df['HSD3_Sold']
total_mismatch = (abs(df['Total_Sold'] - df['_calc_total']) > 10).sum()
print(f'Total_Sold mismatches (>10L diff) : {total_mismatch}')

# Rule 2: No negative closing stock
neg_closing = (df['Closing_Stock'] < 0).sum()
print(f'Negative Closing_Stock rows       : {neg_closing}')

# Rule 3: Opening never exceeds 12000
over_capacity = (df['Opening_Stock'] > 12000).sum()
print(f'Opening_Stock > 12000 rows        : {over_capacity}')

# Rule 4: Refill logic correct
refill_mismatch = ((df['Closing_Stock'] < 2000) != (df['Target'] == 1)).sum()
print(f'Refill label mismatches           : {refill_mismatch}')

# Drop temp column
df.drop(columns=['_calc_total'], inplace=True)

print('\n✅ Validation complete')

=== Business Rule Validation ===
Total_Sold mismatches (>10L diff) : 0
Negative Closing_Stock rows       : 0
Opening_Stock > 12000 rows        : 0
Refill label mismatches           : 0

✅ Validation complete


## Step 6 — Add derived features useful for ML

In [11]:
# Stock ratio — how full is the tank as a percentage
df['Stock_Ratio']        = (df['Closing_Stock'] / 12000).round(4)

# Rolling 7-day average of sales (captures weekly trend)
df['Rolling_7d_Sales']   = df['Total_Sold'].rolling(window=7, min_periods=1).mean().round(0)

# Rolling 3-day average of sales
df['Rolling_3d_Sales']   = df['Total_Sold'].rolling(window=3, min_periods=1).mean().round(0)

# Previous day closing stock (lag feature)
df['Prev_Closing']       = df['Closing_Stock'].shift(1).fillna(12000)

# Previous day total sold
df['Prev_Total_Sold']    = df['Total_Sold'].shift(1).fillna(df['Total_Sold'].mean())

# Days since last refill
refill_mask = df['Target'] == 1
df['Days_Since_Refill'] = refill_mask.cumsum()
df['Days_Since_Refill'] = df.groupby('Days_Since_Refill').cumcount()

# Is festival month
df['Is_Festival_Month']  = df['Month'].isin([10, 11]).astype(int)

# Is monsoon month
df['Is_Monsoon_Month']   = df['Month'].isin([6, 7, 8]).astype(int)

# Is summer month
df['Is_Summer_Month']    = df['Month'].isin([3, 4, 5]).astype(int)

print(f'Total features now: {df.shape[1]}')
print(df.columns.tolist())

Total features now: 32
['Date', 'Day', 'Opening_Stock', 'MS_Sold', 'HSD1_Sold', 'HSD2_Sold', 'HSD3_Sold', 'Total_Sold', 'Closing_Stock', 'Cash', 'Online', 'Card', 'Dip', 'Refill_Required', 'Year', 'Month', 'DayOfWeek', 'DayOfYear', 'WeekOfYear', 'Quarter', 'Is_Weekend', 'Day_Num', 'Target', 'Stock_Ratio', 'Rolling_7d_Sales', 'Rolling_3d_Sales', 'Prev_Closing', 'Prev_Total_Sold', 'Days_Since_Refill', 'Is_Festival_Month', 'Is_Monsoon_Month', 'Is_Summer_Month']


## Step 7 — Final summary & save

In [12]:
print('=== Final Dataset Summary ===')
print(f'Total rows         : {len(df)}')
print(f'Total columns      : {df.shape[1]}')
print(f'Date range         : {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Refill days (Yes)  : {df["Target"].sum()} ({df["Target"].mean()*100:.1f}%)')
print(f'No refill (No)     : {(df["Target"]==0).sum()} ({(df["Target"]==0).mean()*100:.1f}%)')
print(f'Avg Total_Sold     : {df["Total_Sold"].mean():.0f} L/day')
print(f'Min Closing_Stock  : {df["Closing_Stock"].min()} L')
print(f'Max Closing_Stock  : {df["Closing_Stock"].max()} L')

# Save clean data
df.to_csv('../data/clean_data.csv', index=False)
print('\n✅ Saved: data/clean_data.csv')

=== Final Dataset Summary ===
Total rows         : 5189
Total columns      : 32
Date range         : 2011-01-01 to 2025-03-16
Refill days (Yes)  : 2180 (42.0%)
No refill (No)     : 3009 (58.0%)
Avg Total_Sold     : 5217 L/day
Min Closing_Stock  : 0 L
Max Closing_Stock  : 8873 L

✅ Saved: data/clean_data.csv
